# Dedekind DSL: Analyst Tier
Join, aggregate, and pivot workflows using analyst-style table operations.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

try:
    from dedekind import SetDef, pivot_table, unpivot_table
except ModuleNotFoundError:
    candidate_roots = [Path.cwd(), *Path.cwd().parents]
    for root in candidate_roots:
        candidate = root / "python"
        if (candidate / "dedekind").exists():
            sys.path.insert(0, str(candidate))
            break
    from dedekind import SetDef, pivot_table, unpivot_table

print("Imported Analyst-tier helpers: SetDef, pivot_table, unpivot_table")

ModuleNotFoundError: No module named 'dedekind'

## Part 1: Analyst Inputs (Tables)
Define compact source tables similar to SQL/Excel workflows.

In [ ]:
df_sales = pd.DataFrame(
    [
        {"date": "2026-01-05", "product_id": "p1", "region": "north", "units": 10, "revenue": 100},
        {"date": "2026-01-07", "product_id": "p2", "region": "north", "units": 5, "revenue": 250},
        {"date": "2026-01-09", "product_id": "p3", "region": "south", "units": 7, "revenue": 140},
        {"date": "2026-02-03", "product_id": "p1", "region": "south", "units": 8, "revenue": 80},
        {"date": "2026-02-10", "product_id": "p2", "region": "north", "units": 4, "revenue": 200},
        {"date": "2026-02-11", "product_id": "p3", "region": "south", "units": 10, "revenue": 200},
    ]
)

df_products = pd.DataFrame(
    [
        {"product_id": "p1", "category": "Hardware"},
        {"product_id": "p2", "category": "Software"},
        {"product_id": "p3", "category": "Accessories"},
    ]
)

df_regions = pd.DataFrame(
    [
        {"region": "north", "segment": "Enterprise"},
        {"region": "south", "segment": "SMB"},
    ]
)

print("sales rows:", len(df_sales))
display(df_sales.head())

## Part 2: Join And Aggregate
Join fact/dimension tables and compute monthly totals by category.

In [ ]:
sales_enriched = (
    df_sales
    .merge(df_products, on="product_id", how="left")
    .merge(df_regions, on="region", how="left")
)

sales_enriched["month"] = pd.to_datetime(sales_enriched["date"]).dt.to_period("M").astype(str)

summary = (
    sales_enriched
    .groupby(["month", "category"], as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        units=("units", "sum"),
    )
)

print("joined+aggregated summary")
display(summary.sort_values(["month", "category"]).reset_index(drop=True))

## Part 3: Pivot/Unpivot Shim (Middle Layer)
Use pandas-backed shims to keep notebook UX analyst-friendly while core pivot support is tracked in issue #170:
https://github.com/vincentk/dedekind/issues/170

In [ ]:
pivoted = pivot_table(
    summary,
    index="month",
    columns="category",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)

unpivotted = unpivot_table(
    pivoted,
    id_vars=["month"],
    var_name="category",
    value_name="revenue",
)

region_values = SetDef.from_dataframe(df_regions, column="region").realize()

print("pivoted revenue matrix")
display(pivoted)

print("unpivoted back to long form")
display(unpivotted.sort_values(["month", "category"]).reset_index(drop=True))

print("realized region values:", region_values)

In [ ]:
pivot_check = pivoted.sort_values("month").reset_index(drop=True)

expected_columns = ["month", "Accessories", "Hardware", "Software"]
assert list(pivot_check.columns) == expected_columns, pivot_check.columns.tolist()

expected_rows = [
    {"month": "2026-01", "Accessories": 140, "Hardware": 100, "Software": 250},
    {"month": "2026-02", "Accessories": 200, "Hardware": 80, "Software": 200},
]
assert pivot_check.to_dict(orient="records") == expected_rows

assert sorted(region_values) == ["north", "south"]

print("analyst flow verified: join + aggregate + pivot + unpivot")
print("notebook-03-ok")